In [4]:
!pip install pyspark psycopg2 sqlalchemy pandas

In [5]:
from pyspark.sql import SparkSession
import psycopg2
import pandas as pd
from pyspark.sql.functions import col, when, trim, lower, upper, regexp_replace, to_date, concat_ws, monotonically_increasing_id, year, month
from sqlalchemy import create_engine


In [6]:
spark =  SparkSession.builder\
         .appName("DealWithus")\
         .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0")\
         .getOrCreate()

In [7]:
df = spark.read.csv(r"C:\Users\Admin\Desktop\10Alytics\DealWithUs\DealWithUs\Raw_Data\dealwithus_raw_data.csv", header = True, inferSchema = True)

In [ ]:
df.show(5)

+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|     CustomerPhone|             City|             Country|       ProductName| Category|FirstName|Last Name|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+
|  O0094|    P0001|       1|     C1954|2024-12-02|       PayPal|   1491.0|    Pending|    8476.45|griffinmichelle@e...|     (808)782-4405|       Emilymouth|Northern Mariana ...|Interesting Tablet|Computers|   Kelsey|   Burton|
|  O0312|    P0001|       2|     C0827|2025-01-20|  Credit Card|   1491.0|    Shipped|     2

In [9]:
df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- CustomerEmail: string (nullable = true)
 |-- CustomerPhone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- Last Name: string (nullable = true)



In [10]:
for column in df.columns :
    print(f"{column}: {df.filter(df[column].isNull()).count()} Null values") 

OrderID: 0 Null values
ProductID: 0 Null values
Quantity: 0 Null values
CustomerID: 0 Null values
OrderDate: 0 Null values
PaymentMethod: 172 Null values
UnitPrice: 0 Null values
OrderStatus: 0 Null values
TotalAmount: 0 Null values
CustomerEmail: 0 Null values
CustomerPhone: 0 Null values
City: 130 Null values
Country: 0 Null values
ProductName: 0 Null values
Category: 0 Null values
FirstName: 0 Null values
Last Name: 0 Null values


In [11]:
df_clean = df.fillna({'PaymentMethod': 'Not Provided',
                      'City': 'Not Provided',
                     })

In [12]:
for column in df_clean.columns :
    print(f"{column}: {df_clean.filter(df_clean[column].isNull()).count()} Null values") 

OrderID: 0 Null values
ProductID: 0 Null values
Quantity: 0 Null values
CustomerID: 0 Null values
OrderDate: 0 Null values
PaymentMethod: 0 Null values
UnitPrice: 0 Null values
OrderStatus: 0 Null values
TotalAmount: 0 Null values
CustomerEmail: 0 Null values
CustomerPhone: 0 Null values
City: 0 Null values
Country: 0 Null values
ProductName: 0 Null values
Category: 0 Null values
FirstName: 0 Null values
Last Name: 0 Null values


In [13]:
df_clean = df_clean.withColumn('FirstName', trim(col('FirstName')))
df_clean = df_clean.withColumn('Last Name', trim(col('Last Name')))
df_clean = df_clean.withColumn ('PaymentMethod', trim(col('PaymentMethod')))
df_clean = df_clean.withColumn('CustomerEmail', trim(lower(col('CustomerEmail'))))
df_clean = df_clean.withColumn('CustomerEmail', regexp_replace(col('CustomerEmail'), " ", ""))
df_clean = df_clean.withColumn('Category', trim(col('Category')))
df_clean = df_clean.withColumn('CustomerPhone', regexp_replace(col('CustomerPhone'), "[^0-9]", ""))
df_clean = df_clean.withColumn('CustomerName', concat_ws(" ", col('FirstName'), col('Last Name')))


In [14]:
df_clean.show(10)

+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+----------------+-----------------+--------------------+------------------+---------+---------+---------+----------------+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|   CustomerPhone|             City|             Country|       ProductName| Category|FirstName|Last Name|    CustomerName|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+----------------+-----------------+--------------------+------------------+---------+---------+---------+----------------+
|  O0094|    P0001|       1|     C1954|2024-12-02|       PayPal|   1491.0|    Pending|    8476.45|griffinmichelle@e...|      8087824405|       Emilymouth|Northern Mariana ...|Interesting Tablet|Computers|   Kelsey|   Burton|   Kelsey Burton|
|  O0312|    P0001|       2|    

In [15]:
df_clean.dropDuplicates()

DataFrame[OrderID: string, ProductID: string, Quantity: int, CustomerID: string, OrderDate: date, PaymentMethod: string, UnitPrice: double, OrderStatus: string, TotalAmount: double, CustomerEmail: string, CustomerPhone: string, City: string, Country: string, ProductName: string, Category: string, FirstName: string, Last Name: string, CustomerName: string]

In [16]:
df_clean.columns

['OrderID',
 'ProductID',
 'Quantity',
 'CustomerID',
 'OrderDate',
 'PaymentMethod',
 'UnitPrice',
 'OrderStatus',
 'TotalAmount',
 'CustomerEmail',
 'CustomerPhone',
 'City',
 'Country',
 'ProductName',
 'Category',
 'FirstName',
 'Last Name',
 'CustomerName']

### Normalization


In [17]:
customer_dim = df_clean.select('CustomerName', 'CustomerEmail', 'CustomerPhone', 'City', 'Country').dropDuplicates()
customer_dim = customer_dim.withColumn('CustomerID', monotonically_increasing_id()+1)
customer_dim = customer_dim.select('CustomerID', 'CustomerName', 'CustomerEmail', 'CustomerPhone', 'City', 'Country')



In [18]:
customer_dim.show(5)

+----------+--------------+--------------------+----------------+----------------+--------------------+
|CustomerID|  CustomerName|       CustomerEmail|   CustomerPhone|            City|             Country|
+----------+--------------+--------------------+----------------+----------------+--------------------+
|         1|    Jacob Carr|petersonamber@exa...|  69696689319648|       Lake John|               Korea|
|         2|Christy Willis|ayalagregory@exam...|1838826089618331|    Bautistabury|    Saint Barthelemy|
|         3| Kenneth Gomez|danielcline@examp...|   0015863776786|       Edgarland|Holy See (Vatican...|
|         4|  Bryan Hanson|amberhall@example...| 795823090646520|       Laurabury|              Brazil|
|         5|    Samuel Lee|yesenia10@example...|  30294534400663|New Williammouth|            Anguilla|
+----------+--------------+--------------------+----------------+----------------+--------------------+
only showing top 5 rows


In [19]:
product_dim = df_clean.select('ProductID','ProductName', 'Category', 'UnitPrice').dropDuplicates()


In [20]:
product_dim.show(5)

+---------+------------------+-----------+---------+
|ProductID|       ProductName|   Category|UnitPrice|
+---------+------------------+-----------+---------+
|    P0055|       None Laptop|      Audio|   456.14|
|    P0081|     Soldier Phone|      Audio|  1389.31|
|    P0058|     Dinner Laptop|      Audio|   991.25|
|    P0134|        Far Laptop|Accessories|  1068.26|
|    P0005|Morning Headphones|Electronics|  1483.93|
+---------+------------------+-----------+---------+
only showing top 5 rows


In [21]:
order_fact = df_clean.select('OrderID', 'OrderDate', 'OrderStatus', 'PaymentMethod', 'TotalAmount', 'CustomerEmail')
order_fact = order_fact.join(customer_dim, on ='CustomerEmail', how = 'left')
order_fact = order_fact.select('OrderID','CustomerID', 'CustomerEmail','OrderDate', 'OrderStatus', 'PaymentMethod', 'TotalAmount')

In [22]:
order_fact.show(5)

+-------+----------+--------------------+----------+-----------+-------------+-----------+
|OrderID|CustomerID|       CustomerEmail| OrderDate|OrderStatus|PaymentMethod|TotalAmount|
+-------+----------+--------------------+----------+-----------+-------------+-----------+
|  O0094|      1940|griffinmichelle@e...|2024-12-02|    Pending|       PayPal|    8476.45|
|  O0312|       743|adamvelasquez@exa...|2025-01-20|    Shipped|  Credit Card|     2982.0|
|  O0483|      1359|  bsimon@example.com|2025-06-10|    Shipped|       PayPal|    6110.58|
|  O0556|      2074|kellyhenry@exampl...|2025-10-15|    Pending|   Debit Card|    6364.88|
|  O0875|       520|jessicavelazquez@...|2025-01-04|  Cancelled|  Credit Card|    4784.71|
+-------+----------+--------------------+----------+-----------+-------------+-----------+
only showing top 5 rows


In [23]:
order_item_fact = df_clean.select('OrderID', 'ProductID', 'TotalAmount','Quantity')
order_item_fact = order_item_fact.join(product_dim, on = 'ProductID', how = 'left')
order_item_fact = order_item_fact.select('OrderID', 'ProductID', 'ProductName', 'Category', 'UnitPrice','TotalAmount','Quantity')

In [24]:
order_item_fact.show(5)

+-------+---------+------------------+---------+---------+-----------+--------+
|OrderID|ProductID|       ProductName| Category|UnitPrice|TotalAmount|Quantity|
+-------+---------+------------------+---------+---------+-----------+--------+
|  O0094|    P0001|Interesting Tablet|Computers|   1491.0|    8476.45|       1|
|  O0312|    P0001|Interesting Tablet|Computers|   1491.0|     2982.0|       2|
|  O0483|    P0001|Interesting Tablet|Computers|   1491.0|    6110.58|       2|
|  O0556|    P0001|Interesting Tablet|Computers|   1491.0|    6364.88|       1|
|  O0875|    P0001|Interesting Tablet|Computers|   1491.0|    4784.71|       1|
+-------+---------+------------------+---------+---------+-----------+--------+
only showing top 5 rows


### Loading


In [31]:
url = "jdbc:postgresql://localhost:5432/dealwithus"
user = "postgres"
password = "postgres"

def load_to_postgres(df, table_name):
    df.write.format("jdbc")\
    .option("url", url)\
    .option("dbtable", table_name)\
    .option("user", user)\
    .option("password", password)\
    .option("driver", "org.postgresql.Driver")\
    .mode("overwrite")\
    .save()



In [32]:
load_to_postgres(customer_dim, "silver.customer_dim")
load_to_postgres(product_dim, "silver.product_dim")
load_to_postgres(order_fact, "silver.order_fact")
load_to_postgres(order_item_fact, "silver.order_item_fact")